In [20]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Dict, List, Optional
import json
import dotenv
import os

In [19]:
dotenv.load_dotenv()

True

In [23]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [2]:
from ontology import (
    ONTOLOGY
)

entity types: ['NewsArticle', 'Organization', 'Person', 'Place', 'Product']
relation types: ['MENTIONS', 'ANNOUNCED', 'PARTNERED_WITH', 'LOCATED_IN', 'ANALYZED_BY']
attribute schema: {'NewsArticle': {'mentioned_date': 'date'}, 'Organization': {'market_cap_trillion_krw': 'number', 'revenue_target_trillion_krw': 'number', 'stock_change_percent': 'number'}, 'Product': {'product_category': 'string'}, 'Person': {'role': 'string'}, 'Place': {'admin_area': 'string'}}


In [3]:
ontology_text = json.dumps(
    ONTOLOGY,
    ensure_ascii=False,
    indent=2
)

## 1_Crawling의 결과

In [4]:
inputs = {
  'Apple': [
    '삼성전자는 Apple에 OLED 패널 공급을 하는 주요 협력사인 Samsung Display를 통해 협력 관계에 있다.',
    'Apple의 Foldable iPhone용 OLED 패널은 Samsung Display가 공급하며, Apple은 3년간 독점 공급 계약 조건하에 이 패널을 공급받는다.'
  ],
  'SK hynix': [
    '삼성전자는 SK hynix와 메모리 DRAM/NAND 분야에서 경쟁 관계에 있다.',
    'AI 메모리 수요 급증에 따라 양사 간 메모리 공급 및 가격에 영향이 있다.'
  ],
  'LG Display': [
    '삼성전자는 LG Display와 OLED 디스플레이 부문에서 경쟁 관계에 있다.',
    'LG Display는 Apple 등 주요 고객에 OLED 패널을 공급하며 OLED 시장에서 양자 간 경쟁 중이다.'
  ],
  'TSMC': [
    '삼성전자는 TSMC와 파운드리 부문에서 직접 경쟁 관계에 있다.'
  ],
  'BOE': [
    '삼성 Display와 BOE 간 OLED 패널 분야에서 경쟁 관계에 있으며, OLED 특허 및 무역 비밀 분쟁이 있었고 합의로 해결되었다.'
  ]
}

In [24]:
llm = ChatOpenAI(
    model='gpt-5-nano',
    api_key=OPENAI_API_KEY,
    temperature=0.1,
)

# 1. 문장을 임베딩하여 벡터로 변환
결국 객체, 관계, 속성은 문장의 내용에서 추출되는 내용이므로 문장에서의 정보량보다는 적을 것이다.<br>
따라서 문장 자체를 임베딩해서 저장해두면 검색은 문장 임베딩으로, 그래프 탐색은 객체 관계로 진행한다.

In [21]:
class CompanyRelation(BaseModel):
    object_name: str = Field(
        description="중심 기업과 관계를 맺는 객체 이름"
    )
    relations: List[str] = Field(
        description="중심 기업과 객체 사이의 관계 목록"
    )
    evidence: List[str] = Field(
        description="관계를 뒷받침하는 원문 문장"
    )


class Answer(BaseModel):
    company: str = Field(
        description="웹 크롤링의 중심 대상 기업"
    )
    output: List[CompanyRelation] = Field(
        description="객체별 관계 추출 결과"
    )

In [18]:
prompt = PromptTemplate.from_template("""
당신은 기업 관계를 분석하는 정보 추출기입니다.

중심 기업:
{company}

분석할 문장:
{text}
                                      
객체와 관계 정의 온톨로지:
{ontology}
                                      
규칙:
1. 중심 기업과 다른 객체 사이의 관계를 추출하세요.
2. 객체와 관계 유형은 주어진 온톨로지를 따르세요.
3. 같은 기업에 여러 관계가 있으면 모두 보존하세요.
4. 입력 문장으로 확인할 수 없는 관계는 만들지 마세요.
5. 관계마다 근거가 되는 원문 문장을 evidence에 넣으세요.
6. 딕셔너리 key 외의 기업이 문장에 나오면 해당 기업도 추출하세요.
""")

In [22]:
structured_llm = llm.with_structured_output(
    Answer,
    method="json_schema",
    strict=True
)

In [23]:
chain = prompt | structured_llm

In [24]:
text = "\n\n".join(
    f"[관련 기업 후보: {company}]\n" + "\n".join(sentence)
    for company, sentence in inputs.items()
)

In [25]:
result = chain.invoke({
    "company": "삼성전자",
    "ontology": ontology_text,
    "text": text,
})

result.model_dump()

{'company': '삼성전자',
 'output': [{'object_name': 'Apple',
   'relations': ['PARTNERED_WITH'],
   'evidence': ['삼성전자는 Apple에 OLED 패널 공급을 하는 주요 협력사인 Samsung Display를 통해 협력 관계에 있다.']},
  {'object_name': 'SK hynix',
   'relations': ['MENTIONS'],
   'evidence': ['삼성전자는 SK hynix와 메모리 DRAM/NAND 분야에서 경쟁 관계에 있다.',
    'AI 메모리 수요 급증에 따라 양사 간 메모리 공급 및 가격에 영향이 있다.']},
  {'object_name': 'LG Display',
   'relations': ['MENTIONS'],
   'evidence': ['삼성전자는 LG Display와 OLED 디스플레이 부문에서 경쟁 관계에 있다.',
    'LG Display는 Apple 등 주요 고객에 OLED 패널을 공급하며 OLED 시장에서 양자 간 경쟁 중이다.']},
  {'object_name': 'TSMC',
   'relations': ['MENTIONS'],
   'evidence': ['삼성전자는 TSMC와 파운드리 부문에서 직접 경쟁 관계에 있다.']},
  {'object_name': 'BOE',
   'relations': ['MENTIONS'],
   'evidence': ['삼성 Display와 BOE 간 OLED 패널 분야에서 경쟁 관계에 있으며, OLED 특허 및 무역 비밀 분쟁이 있었고 합의로 해결되었다.']}]}